In [2]:
#preprocess scanned pdf or pdf2

In [12]:
import cv2
import numpy as np
from pdf2image import convert_from_path
from scipy.ndimage import rotate
from PIL import Image
import pytesseract

# Function to correct skew
def correct_skew(image, delta=1, limit=12):
    def determine_score(arr, angle):
        data = rotate(arr, angle, reshape=False, order=0)
        histogram = np.sum(data, axis=1, dtype=float)
        score = np.sum((histogram[1:] - histogram[:-1]) ** 2, dtype=float)
        return histogram, score

    gray = cv2.cvtColor(np.array(image), cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 41, 15)
    scores = []
    angles = np.arange(-limit, limit + delta, delta)
    
    for angle in angles:
        histogram, score = determine_score(thresh, angle)
        scores.append(score)
    
    best_angle = angles[scores.index(max(scores))]
    return best_angle

# Function to deskew image
def deskew_image(image, angle):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return rotated

# Function to detect text orientation using Tesseract
def detect_orientation(image):
    image = np.array(image)
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    osd = pytesseract.image_to_osd(rgb_image)
    rotate_angle = 0
    if "Rotate: 180" in osd:
        rotate_angle = 180
    return rotate_angle

# Path to the PDF file
pdf_path = '/home/emanmunir/detection/test5/scan-rotated.pdf'
# Convert PDF to images
images = convert_from_path(pdf_path)

# Process images and save to a new PDF
pdf_images = []
for i, image in enumerate(images):
    rotate_angle = detect_orientation(image)
    if rotate_angle == 180:
        image = image.rotate(180, expand=True)
    angle = correct_skew(image)
    deskewed_image = deskew_image(np.array(image), angle)
    final_image = Image.fromarray(deskewed_image)
    pdf_images.append(final_image.convert('RGB'))

# Save all images into a single PDF
pdf_images[0].save('/home/emanmunir/detection/test5/scan-deskewed.pdf', save_all=True, append_images=pdf_images[1:])


In [3]:
#use the deskewed pdf to remove white margins


In [7]:
#code for removing white margin
import fitz  # PyMuPDF
import numpy as np

def remove_white_margins(input_pdf, output_pdf, margin_threshold=5):
    doc = fitz.open(input_pdf)
    for page_num in range(len(doc)):
        page = doc[page_num]
        pix = page.get_pixmap(alpha=False)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
        
        # Convert to grayscale if needed
        if img.ndim == 3:
            img = np.mean(img, axis=2)
        
        # Find bounding box of non-white pixels
        mask = img < 255 - margin_threshold
        coords = np.argwhere(mask)
        if coords.size == 0:
            continue
        x0, y0 = coords.min(axis=0)
        x1, y1 = coords.max(axis=0) + 1
        
        # Set crop box
        rect = fitz.Rect(y0, x0, y1, x1)
        page.set_cropbox(rect)
        
    doc.save(output_pdf)

input_pdf = "/home/emanmunir/detection/test5/scan-deskewed.pdf"
scan_preprocessed = "/home/emanmunir/detection/test5/scan-preprocessed.pdf"
remove_white_margins(input_pdf, scan_preprocessed)


In [11]:
import fitz  # PyMuPDF
import cv2
import numpy as np
import os
from PIL import Image
import imagehash

def extract_annotations(doc, color='red'):
    annotations_info = []
    color_map = {'red': ([0, 0, 250], [10, 10, 255])}  # Define BGR ranges for red

    for page_number in range(len(doc)):
        page = doc[page_number]
        pix = page.get_pixmap()

        if pix.samples:
            image = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            lower_color, upper_color = color_map[color] if color in color_map else ([0, 0, 0], [255, 255, 255])
            lower_red = np.array(lower_color)
            upper_red = np.array(upper_color)

            mask = cv2.inRange(image, lower_red, upper_red)
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            for rect in [cv2.boundingRect(cnt) for cnt in contours]:
                x0, y0, w, h = rect
                x1, y1 = x0 + w, y0 + h
                annotations_info.append((page_number + 1, x0, y0, x1, y1))

    return annotations_info

def remove_border(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        # Find the largest contour which should be the border
        contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(contour)
        img = img[y:y+h, x:x+w]
    return img, (x, y, w, h)

def extract_images(doc, annotations_info, output_folder, remove_border_flag=False):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    output_images = []
    new_annotations_info = []
    for idx, (page_number, x0, y0, x1, y1) in enumerate(annotations_info):
        page = doc.load_page(page_number - 1)
        clip = fitz.Rect(x0, y0, x1, y1)
        pix = page.get_pixmap(clip=clip)
        img = cv2.imdecode(np.frombuffer(pix.tobytes(), dtype=np.uint8), cv2.IMREAD_COLOR)
        if remove_border_flag:
            img, (border_x, border_y, border_w, border_h) = remove_border(img)
            new_x0 = x0 + border_x
            new_y0 = y0 + border_y
            new_x1 = x0 + border_x + border_w
            new_y1 = y0 + border_y + border_h
            new_annotations_info.append((page_number, new_x0, new_y0, new_x1, new_y1))
        else:
            new_annotations_info.append((page_number, x0, y0, x1, y1))
        output_images.append(img)
        img_path = os.path.join(output_folder, f"extracted_image_{idx + 1}.png")
        cv2.imwrite(img_path, img)
    return output_images, new_annotations_info

def adjust_annotations_for_pdf2(original_annotations, source_rect, target_rect):
    adjusted_annotations = []
    x_scale = target_rect.width / source_rect.width
    y_scale = target_rect.height / source_rect.height
    for (page_number, x0, y0, x1, y1) in original_annotations:
        new_x0 = x0 * x_scale
        new_y0 = y0 * y_scale
        new_x1 = x1 * x_scale
        new_y1 = y1 * y_scale
        adjusted_annotations.append((page_number, new_x0, new_y0, new_x1, new_y1))
    return adjusted_annotations

def compute_hash_similarity(img1, img2):
    img1_pil = Image.fromarray(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
    img2_pil = Image.fromarray(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
    hash1 = imagehash.phash(img1_pil)
    hash2 = imagehash.phash(img2_pil)
    similarity = 1 - (hash1 - hash2) / len(hash1.hash) ** 2
    return similarity

input_path1 = '/home/emanmunir/detection/test2/pdf to image/small-pdf-boxes (1).pdf'
input_path2 = "/home/emanmunir/detection/test5/scan-preprocessed.pdf"
doc1 = fitz.open(input_path1)
doc2 = fitz.open(input_path2)

annotations_info1 = extract_annotations(doc1)

output_folder1 = "/home/emanmunir/detection/test5/pdf1"
output_folder2 = "/home/emanmunir/detection/test5/pdf2"
output_images1, new_annotations_info1 = extract_images(doc1, annotations_info1, output_folder1, remove_border_flag=True)

source_rect = doc1[0].rect
target_rect = doc2[0].rect
adjusted_annotations_info2 = adjust_annotations_for_pdf2(annotations_info1, source_rect, target_rect)
output_images2, _ = extract_images(doc2, adjusted_annotations_info2, output_folder2, remove_border_flag=True)

similarities = []
for img1, img2 in zip(output_images1, output_images2):
    score = compute_hash_similarity(img1, img2)
    similarities.append(score)

print(f"Annotations info: {annotations_info1}, Similarities: {similarities}")


Annotations info: [(1, 83, 336, 219, 406), (1, 317, 59, 542, 311), (2, 111, 370, 312, 562), (2, 324, 294, 442, 377), (3, 88, 478, 227, 627), (3, 349, 408, 534, 613), (3, 348, 70, 494, 210), (4, 330, 450, 469, 592), (4, 324, 229, 503, 368), (4, 85, 76, 263, 211), (5, 371, 135, 513, 257), (6, 119, 237, 297, 455), (6, 364, 152, 489, 311), (7, 90, 636, 251, 724), (7, 289, 634, 520, 737), (7, 247, 481, 384, 552), (7, 405, 477, 555, 570), (7, 73, 348, 571, 471), (7, 448, 109, 541, 182), (8, 370, 252, 548, 355), (8, 154, 252, 327, 351)], Similarities: [0.5, 0.6875, 0.59375, 0.46875, 0.625, 0.71875, 0.5, 0.5625, 0.65625, 0.65625, 0.5625, 0.65625, 0.5, 0.46875, 0.515625, 0.5625, 0.59375, 0.5, 0.5, 0.5, 0.46875]
